# EDSS 취업통계 코호트 연도 감사

## tl;dr

2010–2022년 파일 13개는 실제로 2010–2020년 졸업 코호트 11개를 포함한다. 2014 파일은 같은 2014 코호트의 6월 1일 파동이라 제외하고 2015 파일의 12월 31일 파동을 선택한다. 2021·2022년은 같은 2020 코호트의 학교 집계가 완전히 같아 2022년을 제외한다. 최종 선택 원천은 11개다.

## Context & Methods

이 노트북은 `scripts/audit_edss_employment_cohort_years.py`가 제한 DuckDB를 읽기 전용으로 감사해 만든 CSV와 JSON을 독립 검산한다.

### Key Assumptions

- 7–12월 졸업은 다음 해 2월 졸업과 같은 코호트로, 1–6월 졸업은 해당 연도 코호트로 본다.
- 행 수 기준 최빈 코호트를 파일의 대표 코호트로 사용한다.
- 개인식별번호, 회사명, 논문명과 개별 원천 행은 노트북에 적재하거나 출력하지 않는다.
- `_panel_year`는 원본 제공 라벨이며 공식 조사연도나 졸업 코호트로 직접 해석하지 않는다.

## Data

### 1. Load audit outputs

In [1]:
import csv
import json
from pathlib import Path

from IPython.display import Markdown, display


def find_repo_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data/metadata/edss_employment_cohort_year_audit.csv").exists():
            return candidate
    raise FileNotFoundError("run this notebook from the EDSS repository")


repo_root = find_repo_root()
csv_path = repo_root / "data/metadata/edss_employment_cohort_year_audit.csv"
json_path = repo_root / "data/metadata/edss_employment_cohort_year_audit.json"

with csv_path.open(encoding="utf-8-sig", newline="") as handle:
    year_rows = list(csv.DictReader(handle))
summary = json.loads(json_path.read_text(encoding="utf-8"))

print(f"source years: {len(year_rows)}")
print(f"source rows: {summary['source_row_count']:,}")
print(f"inferred cohorts: {summary['inferred_cohort_year_count']}")
print(f"audit status: {summary['status']}")

source years: 13
source rows: 7,277,987
inferred cohorts: 11
audit status: complete


## Results

### 2. File years do not form a cohort-year series

In [2]:
def markdown_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "|" + "|".join(["---"] * len(headers)) + "|"]
    lines.extend("| " + " | ".join(map(str, row)) + " |" for row in rows)
    return "\n".join(lines)


mapping_rows = []
for row in year_rows:
    mapping_rows.append(
        (
            row["source_year"],
            row["inferred_cohort_year"],
            row["observation_reference_date_basis"],
            f"{float(row['primary_august_february_share']) * 100:.3f}%",
            row["source_minus_cohort_years"],
            row["cohort_use_status"],
        )
    )
display(Markdown(markdown_table(
    ["source year", "cohort", "reference basis", "Aug+Feb share", "offset", "use status"],
    mapping_rows,
)))

| source year | cohort | reference basis | Aug+Feb share | offset | use status |
|---|---|---|---|---|---|
| 2010 | 2010 | june_1 | 98.047% | 0 | eligible_unique_cohort |
| 2011 | 2011 | june_1 | 98.792% | 0 | eligible_unique_cohort |
| 2012 | 2012 | june_1 | 98.953% | 0 | eligible_unique_cohort |
| 2013 | 2013 | june_1 | 99.600% | 0 | eligible_unique_cohort |
| 2014 | 2014 | june_1 | 99.871% | 0 | exclude_transition_june_wave |
| 2015 | 2014 | december_31 | 99.871% | 1 | eligible_transition_december_wave |
| 2016 | 2015 | december_31 | 99.769% | 1 | eligible_unique_cohort |
| 2017 | 2016 | december_31 | 99.568% | 1 | eligible_unique_cohort |
| 2018 | 2017 | december_31 | 99.803% | 1 | eligible_unique_cohort |
| 2019 | 2018 | december_31 | 99.775% | 1 | eligible_unique_cohort |
| 2020 | 2019 | december_31 | 99.108% | 1 | eligible_unique_cohort |
| 2021 | 2020 | december_31 | 99.178% | 1 | eligible_first_of_exact_repeat |
| 2022 | 2020 | december_31 | 99.178% | 2 | exclude_exact_repeat |

### 3. The two repeated cohorts have different quality outcomes

In [3]:
comparison_rows = []
for item in summary["same_cohort_comparisons"]:
    comparison_rows.append(
        (
            item["inferred_cohort_year"],
            f"{item['previous_source_year']}→{item['current_source_year']}",
            item["shared_open_id_count"],
            item["exact_school_aggregate_match_count"],
            f"{item['exact_school_aggregate_match_share'] * 100:.3f}%",
            item["graduation_month_distribution_equal"],
            item["complete_exact_repeat"],
        )
    )
display(Markdown(markdown_table(
    ["cohort", "source pair", "shared IDs", "exact", "exact share", "month distribution equal", "complete repeat"],
    comparison_rows,
)))

| cohort | source pair | shared IDs | exact | exact share | month distribution equal | complete repeat |
|---|---|---|---|---|---|---|
| 2014 | 2014→2015 | 552 | 20 | 3.623% | False | False |
| 2020 | 2021→2022 | 537 | 537 | 100.000% | True | True |

### 4. Recompute the decision-critical checks

In [4]:
assert len(year_rows) == 13
assert sum(int(row["source_row_count"]) for row in year_rows) == 7_277_987
assert all(int(row["graduation_month_blank_count"]) == 0 for row in year_rows)
assert all(int(row["graduation_month_invalid_count"]) == 0 for row in year_rows)
assert summary["inferred_cohort_year_count"] == 11
assert summary["repeated_cohort_groups"] == {
    "2014": ["2014", "2015"],
    "2020": ["2021", "2022"],
}
assert summary["status"] == "complete"
assert summary["cohort_analysis_eligible_source_year_count"] == 11

by_year = {row["source_year"]: row for row in year_rows}
assert by_year["2014"]["cohort_use_status"] == "exclude_transition_june_wave"
assert by_year["2014"]["observation_reference_date"] == "2014-06-01"
assert by_year["2015"]["cohort_use_status"] == "eligible_transition_december_wave"
assert by_year["2015"]["observation_reference_date"] == "2014-12-31"
assert by_year["2021"]["inferred_cohort_year"] == "2020"
assert by_year["2021"]["cohort_use_status"] == "eligible_first_of_exact_repeat"
assert by_year["2022"]["cohort_use_status"] == "exclude_exact_repeat"
assert by_year["2022"]["same_cohort_exact_school_aggregate_match_count"] == "537"
assert by_year["2022"]["same_cohort_month_distribution_equal_to_previous"] == "true"

print("All cohort-axis reconciliation checks passed.")

All cohort-axis reconciliation checks passed.


### 5. Validate the final DuckDB cohort mart

In [5]:
import duckdb

db_path = repo_root / "data/processed/edss/restricted/edss_all.duckdb"
connection = duckdb.connect(str(db_path), read_only=True)
mart_stats = connection.execute("""
    SELECT count(*),
           count(DISTINCT (employment_cohort_year, 개방ID)),
           count(DISTINCT employment_cohort_year),
           sum(source_record_count),
           count(*) FILTER (WHERE employment_source_panel_year IN ('2014', '2022'))
    FROM analysis.employment_cohort_school_2010_2020
""").fetchone()
join_stats = connection.execute("""
    SELECT count(*), count(DISTINCT (_panel_year, 개방ID)),
           count(*) FILTER (WHERE _employment_cohort_exists = 'true')
    FROM analysis.school_year_core_with_employment_cohort_2010_2020
""").fetchone()
selected_transition = connection.execute("""
    SELECT employment_source_panel_year, employment_reference_date
    FROM meta.employment_cohort_source_map
    WHERE employment_cohort_year = '2014'
""").fetchone()
connection.close()

assert mart_stats == (5_969, 5_969, 11, 6_167_230, 0)
assert join_stats == (20_226, 20_226, 5_969)
assert selected_transition == ('2015', '2014-12-31')
print("Final cohort mart: 5,969 unique keys across 11 cohorts")
print("Selected source records: 6,167,230")
print("Core join: 20,226 rows, no expansion")

Final cohort mart: 5,969 unique keys across 11 cohorts
Selected source records: 6,167,230
Core join: 20,226 rows, no expansion


## Takeaways

- `_panel_year`를 졸업 코호트 연도로 사용하면 2015년 이후 결과가 한 해 잘못 표시된다.
- 2014 코호트는 2014 파일의 6월 1일 파동을 제외하고 2015 파일의 12월 31일 파동을 선택한다.
- 2020 코호트는 2021년 파일을 보존하고, 완전 반복인 2022년 파일은 코호트·시계열 분석에서 제외한다.
- 최종 DuckDB 코호트 마트는 5,969개 유일키와 선택 원천 6,167,230행을 정확히 보존한다.